# Zero-shot + Llama SEA-LION v3 8B
Run zero-shot inference on the full qas_test.json dataset.

In [ ]:
%pip install -q "vllm>=0.8.3" "bitsandbytes>=0.45.3" "jsonschema>=4.0" rouge-score nltk

In [ ]:
MODEL_ID = "aisingapore/Llama-SEA-LION-v3-8B-IT"
SOURCE_MODE = "git"
KAGGLE_REPO_ROOT = "/kaggle/input/poma-repo/POMA"  # only used for SOURCE_MODE=dataset
REPO_URL = "https://github.com/NgDinhKhoi0709/POMA.git"
REVISION = "main"
OUTPUT_ROOT = "/kaggle/working/poma_sea_lion"


In [ ]:
import os, subprocess, sys
from pathlib import Path
repo = Path(KAGGLE_REPO_ROOT)
if SOURCE_MODE == "git":
    repo = Path("/kaggle/working/POMA")
    if (repo / ".git").exists():
        subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", REVISION], check=True)
        subprocess.run(["git", "-C", str(repo), "checkout", "--detach", "FETCH_HEAD"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REVISION, REPO_URL, str(repo)], check=True)
assert all((repo / "dataset" / name).exists() for name in ["qas_dev.json", "qas_test.json", "table.json"])
os.environ.update({"POMA_LLM_MODEL": "local/sea-lion-v3-8b-it", "POMA_LOCAL_MODEL_ID": MODEL_ID, "POMA_LOCAL_BACKEND": "vllm", "POMA_PROMPT_PROFILE": "compact", "POMA_USE_AGENT_HINTS": "true", "POMA_PARALLEL_WORKERS": "1", "POMA_VLLM_GPU_MEMORY_UTILIZATION": "0.90"})


In [ ]:
command = [sys.executable, "scripts/run_sea_lion_kaggle_eval.py", "--repo-root", str(repo), "--output-root", OUTPUT_ROOT, "--phase", "test", "--mode", "zero_shot", "--model", "local/sea-lion-v3-8b-it"]
subprocess.run(command, cwd=repo, check=True)


In [ ]:
import shutil
results_dir = Path(OUTPUT_ROOT)
archive_path = shutil.make_archive("/kaggle/working/poma_sea_lion_results", "zip", results_dir)
print(f"Saved results archive: {archive_path}")
